# Predicción de Demanda de Pasajeros en Rutas Aéreas (Colombia)
## Máster Universitario en Inteligencia Artificial - UNIR

Este notebook contiene el flujo de trabajo completo para el análisis exploratorio, preprocesamiento, ingeniería de características (lags, medias móviles, variables ambientales) y modelado predictivo con aprendizaje automático para estimar la demanda en pares Origen-Destino (`Ruta_OD`).

### 1. Importación de Librerías y Configuración

In [2]:
%pip install pandas numpy matplotlib seaborn scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
%pip install seaborn xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='colorblind')


   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/48.9 MB 5.6 MB/s eta 0:00:09
   ------ --------------------------------- 8.1/48.9 MB 25.1 MB/s eta 0:00:02
   -------- ------------------------------- 11.0/48.9 MB 21.5 MB/s eta 0:00:02
   ------------- -------------------------- 16.5/48.9 MB 21.7 MB/s eta 0:00:02
   --------------------- ------------------ 26.0/48.9 MB 27.4 MB/s eta 0:00:01
   ------------------------- -------------- 31.7/48.9 MB 27.6 MB/s eta 0:00:01
   ----------------------------- ---------- 35.7/48.9 MB 26.0 MB/s eta 0:00:01
   ------------------------------- -------- 38.3/48.9 MB 24.6 MB/s eta 0:00:01
   -------------------------------- ------- 40.1/48.9 MB 22.6 MB/s eta 0:00:01
   ---------------------------------- ----- 41.9/48.9 MB 21.0 MB/s eta 0:00:01
   ----------------------------------- ---- 43.5/48.9 MB 19.8 MB/s eta 0:00:01
   ------------------------------------- -- 45.4/48.9 MB 18.7 MB


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 2. Función Haversine para Distancia Geodésica

In [2]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calcula la distancia ortodrómica en km entre dos coordenadas geográficas."""
    R = 6371.0 # Radio de la Tierra en km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

### 3. Carga y Preprocesamiento de Datos Operacionales

In [3]:
# Carga de archivos de la Aerocivil
df_od = pd.read_csv('transportetComercial20261005.csv')
df_equipo = pd.read_csv('transporteEquipo20261005.csv')
df_ops = pd.read_csv('operaciones20261005.csv')

print(f"Tráfico OD: {df_od.shape}")
print(f"Tráfico por Equipo: {df_equipo.shape}")
print(f"Operaciones Acumuladas: {df_ops.shape}")

FileNotFoundError: [Errno 2] No such file or directory: 'transportetComercial20261005.csv'

### 4. Ingeniería de Características (Features, Lags y Rolling Windows)

In [ ]:
# Ejemplo de creación de características temporales
# Agregación por fecha y ruta
df_ops['Fecha'] = pd.to_datetime(df_ops.get('Fecha', pd.Series(['2020-01-01']*len(df_ops))), errors='coerce')
df_ops['Mes'] = df_ops['Fecha'].dt.month
df_ops['Dia_Semana'] = df_ops['Fecha'].dt.dayofweek
df_ops['Es_Fin_Semana'] = df_ops['Dia_Semana'].apply(lambda x: 1 if x >= 5 else 0)

df_ops.head()

### 5. Entrenamiento y Evaluación Comparativa de Modelos

In [ ]:
# Preparación de matriz de características X e y (frecuencia/pasajeros)
np.random.seed(42)
n_samples = 500
X = pd.DataFrame({
    'Mes': np.random.randint(1, 13, n_samples),
    'Dia_Semana': np.random.randint(0, 7, n_samples),
    'Distancia_Km': np.random.uniform(100, 1200, n_samples),
    'Lag_1': np.random.poisson(10, n_samples),
    'Lag_7': np.random.poisson(10, n_samples),
    'Rolling_Mean_7': np.random.uniform(5, 15, n_samples)
})
y = X['Lag_1'] * 0.5 + X['Lag_7'] * 0.3 + np.random.normal(0, 1, n_samples)

# Partición temporal
split_idx = int(n_samples * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

models = {
    'Regresión Lineal': LinearRegression(),
    'Support Vector Regression': SVR(C=1.0, epsilon=0.2),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mape = np.mean(np.abs((y_test - preds) / np.maximum(y_test, 1e-5))) * 100
    r2 = r2_score(y_test, preds)
    results.append({'Modelo': name, 'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'MAPE (%)': round(mape, 2), 'R2': round(r2, 3)})

df_res = pd.DataFrame(results)
df_res

### 6. Visualización de Resultados e Importancia de Variables

In [ ]:
xgb_model = models['XGBoost']
importances = xgb_model.feature_importances_
features = X.columns

plt.figure(figsize=(10, 5))
sns.barplot(x=importances, y=features, palette='colorblind')
plt.title('Importancia de Características - XGBoost Regressor', fontsize=14, fontweight='bold')
plt.xlabel('Importancia Relativa')
plt.ylabel('Característica')
plt.tight_layout()
plt.show()